# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 colorectal cancer dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

This dataset contains tabular data for cancer survivors with second primary colorectal cancer, including clinical and molecular variables such as demographics, comorbidities, cancer type, treatment history, MSI-H status, and anatomical distribution.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
List all Record Sets along with their `@id`s. For each Record Set, also list the fields and columns with their names and `@id`s.

In [ ]:
# Get all Record Sets in the Croissant metadata using their @id
record_sets = dataset.metadata.record_sets
print(f"Record Sets found: {len(record_sets)}\n")

for record_set in record_sets:
    print(f"Record Set: {record_set.name}")
    print(f"   @id: {record_set.id}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("   Fields:")
        for field in record_set.fields:
            print(f"      - {field.name} (@id: {field.id}) [dataType: {getattr(field, 'data_type', 'N/A')}]")
    if hasattr(record_set, 'columns') and record_set.columns:
        print("   Columns:")
        for col in record_set.columns:
            print(f"      - {col.name} (@id: {col.id}) [dataType: {getattr(col, 'data_type', 'N/A')}]")
    print("\n")

## 3. Data Extraction
Load each record set (as referenced by its `@id`) into a pandas DataFrame.

We will extract all record sets (since this dataset primarily contains a main data table), and display the columns and first few rows.

**Note:** Replace the values in `record_set_ids` below with the results from the previous cell if you want to explore a different set.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Extract data from each record set, keyed by their @id
dataframes = {}

for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Extracted {len(df)} rows from Record Set: {rset_id}")

# Let's examine the first record set:
if len(record_set_ids) > 0:
    selected_record_set_id = record_set_ids[0]
    print(f"\nFields/Columns in record set (@id): {selected_record_set_id}")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll perform some typical data processing and skimming:

- Select a numeric field (by `@id`) and filter records
- Normalize the field
- Group/aggregate by a categorical field (by `@id`)

**Please update the `numeric_field_id` and `group_field_id` below using the proper `@id`s of fields found in the overview.**

In this sample, we pick `Age_at_Second_Primary_CRC_@id` and `Sex_@id` as placeholders. Replace with real values from your record set.

In [ ]:
# Example: edit these using actual @id from your data overview above
# (Replace with the @id of 'Age at Second Primary CRC' and 'Sex' fields/columns)
record_set_id = selected_record_set_id
numeric_field_id = None  # set e.g., 'https://api.app.sen.science/frontiers/7862866/field/Age_at_Second_Primary_CRC'
group_field_id = None    # set e.g., 'https://api.app.sen.science/frontiers/7862866/field/Sex'

# Try to suggest candidate fields based on names in columns
print(f"Available columns in {record_set_id}:")
for col in dataframes[record_set_id].columns:
    print(col)

# Automatically guess numeric and group fields
import re
def guess_field(columns, keywords):
    for k in keywords:
        for c in columns:
            if re.search(k, c, re.IGNORECASE):
                return c
    return columns[0]

if not numeric_field_id:
    numeric_field_id = guess_field(dataframes[record_set_id].columns, ['age', 'interval', 'number', 'duration', 'count'])
if not group_field_id:
    group_field_id = guess_field(dataframes[record_set_id].columns, ['sex', 'gender', 'group', 'status'])

print(f"\nSelected numeric field (@id): {numeric_field_id}")
print(f"Selected group field (@id): {group_field_id}")

# Proceed if numeric field looks suitable
df = dataframes[record_set_id].copy()

if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean()
else:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()

filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the categorical field if it exists
if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df)


## 5. Visualization

Let's visualize the distribution of the selected numeric field and compare grouped values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Bar plot of group mean if grouping available
if group_field_id in df.columns and 'grouped_df' in locals():
    plt.figure(figsize=(6, 4))
    sns.barplot(x=grouped_df.index, y=grouped_df[f'mean_{numeric_field_id}'])
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

- We loaded and explored the dataset using the Croissant schema and `mlcroissant` library.
- We identified available record sets and their fields/columns, extracting their `@id`s for robust access.
- We demonstrated basic exploratory data analysis: filtering and normalization of a numeric field, and aggregating by a categorical field.
- Simple visualizations illustrate data distribution and inter-group comparisons.

Consider adjusting filters, normalizations, or groupings to align with your research questions, and refer to the official `mlcroissant` [documentation](https://mlcroissant.org/) for more advanced data workflows.
